# Diagnostic performance of pre-hospital diagnoses

This notebook computes diagnostic performance for each pre-hospital diagnosis, using the in-hospital diagnosis as the reference standard.

For every diagnosis code, the analysis is performed **one versus rest at the patient level**. The notebook reports:

- true positives, false positives, false negatives and true negatives
- sensitivity
- specificity
- positive predictive value (PPV)
- negative predictive value (NPV)
- accuracy
- 95% confidence intervals for each proportion

The confidence intervals use the **Wilson score method** by default. This is preferable to a simple Wald interval, particularly for rare diagnoses or proportions close to 0 or 1.



Methods:
Diagnostic accuracy of prehospital diagnoses was assessed using the corresponding in hospital diagnoses as the reference standard. For each diagnosis, sensitivity, specificity, positive predictive value, negative predictive value, and accuracy were calculated from patient level one versus rest contingency tables. Ninety five percent confidence intervals were estimated using the Wilson score method.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Configuration

In [ ]:
# Adjust the path if the Excel file is stored elsewhere.
data_path = Path("/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/pediatric_trauma/data/Data_PedRegaTrauma_coded_for_analysis_250417.xlsx")
sheet_name = "All centres cleaned"

# Keep these aligned with dx_accuracy.ipynb
add_poly_trauma_codes = False
polytrauma_threshold = 2
add_in_hospital_accessory_diagnoses = True

# Confidence interval settings
confidence_level = 0.95
ci_method = "wilson"  # use "beta" for exact Clopper-Pearson intervals

if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path.resolve()}. "
        "Update data_path above to point to the Excel file."
    )

In [ ]:
df = pd.read_excel(data_path, sheet_name=sheet_name)

print(f"Rows: {len(df)}")
print(f"Unique patients: {df['Pat ID'].nunique()}")

if df["Pat ID"].duplicated().any():
    raise ValueError(
        "Pat ID is not unique. This notebook assumes one row per patient. "
        "Aggregate or resolve duplicate patient rows before continuing."
    )

In [ ]:
# remove dead patients
df_clean = df[df.NACA != 7]

# clean text dx
df_clean['Main diagnosis in-hospital'] = df_clean['Main diagnosis in-hospital'].replace('Obstrukt.Atemversagen -REA', 10)
df_clean['Main diagnosis in-hospital'] = df_clean['Main diagnosis in-hospital'].replace('C2-Intoxikation', 0)

# remove patients with no in-hospital dx
df_clean = df_clean[(df_clean["Main diagnosis in-hospital"] != 999) & (df_clean["Main diagnosis in-hospital"] != 0)]

In [ ]:
df = df_clean.copy()

## Diagnosis definitions

In [ ]:
dx_code_to_name = {
    0: "No trauma",
    1: "Traumatic brain or cervical spine injury",
    2: "Chest trauma",
    3: "Abdominal trauma",
    4: "Pelvic trauma / lumbar spine",
    5: "Upper extremity trauma",
    6: "Lower extremity trauma",
    7: "Spine injury",
    8: "Facial trauma",
    9: "Polytrauma",
    10: "Drowning/Asphyxia",
    11: "Burns",
}

diagnosis_codes = sorted(dx_code_to_name)

## Build patient-level diagnosis sets

This section deliberately works directly from the original patient rows instead of expanding diagnoses into multiple rows.

That is important for a multi-label reference standard: a patient with one pre-hospital diagnosis and several in-hospital diagnoses must retain **all** in-hospital labels.

In [ ]:
def clean_accessory_diagnosis(value):
    """Reproduce the accessory-diagnosis cleaning used in dx_accuracy.ipynb."""
    if pd.isna(value):
        return ""

    value = str(value).replace(".", ",")
    value = re.sub(r"[^0-9,]", "", value)
    value = value.strip(",")
    value = re.sub(r",+", ",", value)
    return value


def parse_pre_hospital_codes(value):
    """Convert the pre-hospital diagnosis field to a set of integer codes."""
    if pd.isna(value):
        return set()

    text = str(value).strip()

    if text == "<NA>":
        return set()

    # Special recoding already present in dx_accuracy.ipynb
    if text == "Vd. a. Asphiktische REA":
        text = "10"
    elif "Rauchgasvergiftung" in text:
        text = "0"

    return parse_numeric_code_string(text)


def parse_in_hospital_codes(main_value, accessory_value=None):
    """Convert main + optional accessory in-hospital diagnoses to a set of codes."""
    parts = []

    if pd.notna(main_value):
        main_text = str(main_value).strip()

        # Same special handling as dx_accuracy.ipynb
        main_text = main_text.replace("Obstrukt.Atemversagen -REA", "10")

        # C2 intoxication is not one of the trauma diagnosis codes.
        # If an accessory trauma diagnosis is present, retain that trauma diagnosis.
        main_text = main_text.replace("C2-Intoxikation", "")
        main_text = main_text.strip(",")

        if main_text:
            parts.append(main_text)

    if add_in_hospital_accessory_diagnoses:
        accessory_text = clean_accessory_diagnosis(accessory_value)
        if accessory_text:
            parts.append(accessory_text)

    combined = ",".join(parts)
    return parse_numeric_code_string(combined)


def parse_numeric_code_string(value):
    """Parse comma-separated integer diagnosis codes and ignore 999/missing values."""
    if value is None or pd.isna(value):
        return set()

    text = str(value).strip()
    if not text:
        return set()

    codes = set()

    for token in text.split(","):
        token = token.strip()
        if not token:
            continue

        # Only accept tokens that are genuinely numeric diagnosis codes.
        if not re.fullmatch(r"\d+(?:\.0+)?", token):
            continue

        code = int(float(token))

        if code == 999:
            continue

        if code in dx_code_to_name:
            codes.add(code)

    return codes


def maybe_add_polytrauma(codes):
    codes = set(codes)

    if (
        add_poly_trauma_codes
        and len(codes) >= polytrauma_threshold
        and 9 not in codes
    ):
        codes.add(9)

    return codes


patient_rows = []

for _, row in df.iterrows():
    pre_codes = parse_pre_hospital_codes(row["Main diagnosis pre-hospital"])
    in_codes = parse_in_hospital_codes(
        row["Main diagnosis in-hospital"],
        row["Other diagnosis 1"],
    )

    pre_codes = maybe_add_polytrauma(pre_codes)
    in_codes = maybe_add_polytrauma(in_codes)

    patient_rows.append(
        {
            "Pat ID": row["Pat ID"],
            "pre_labels": pre_codes,
            "in_labels": in_codes,
        }
    )

patient_dx = pd.DataFrame(patient_rows).set_index("Pat ID")
patient_dx.head()

## Sanity checks

In [ ]:
all_pre_codes = sorted(set().union(*patient_dx["pre_labels"]))
all_in_codes = sorted(set().union(*patient_dx["in_labels"]))

print("Observed pre-hospital codes:", all_pre_codes)
print("Observed in-hospital codes:", all_in_codes)

unexpected_pre = sorted(set(all_pre_codes) - set(diagnosis_codes))
unexpected_in = sorted(set(all_in_codes) - set(diagnosis_codes))

assert not unexpected_pre, f"Unexpected pre-hospital codes: {unexpected_pre}"
assert not unexpected_in, f"Unexpected in-hospital codes: {unexpected_in}"

# Number of patients carrying each diagnosis in each setting
diagnosis_counts = pd.DataFrame(
    {
        "Diagnosis code": diagnosis_codes,
        "Diagnosis": [dx_code_to_name[c] for c in diagnosis_codes],
        "Pre-hospital positive n": [
            patient_dx["pre_labels"].apply(lambda labels: c in labels).sum()
            for c in diagnosis_codes
        ],
        "In-hospital reference positive n": [
            patient_dx["in_labels"].apply(lambda labels: c in labels).sum()
            for c in diagnosis_codes
        ],
    }
)

display(diagnosis_counts)

## Diagnostic performance with 95% confidence intervals

For each diagnosis:

- **Sensitivity** = TP / (TP + FN)
- **Specificity** = TN / (TN + FP)
- **PPV** = TP / (TP + FP)
- **NPV** = TN / (TN + FN)
- **Accuracy** = (TP + TN) / N

The confidence interval for each metric uses the same numerator and denominator as that metric. If a denominator is zero, the metric and its confidence interval are reported as missing rather than forcing a value of zero.

In [ ]:
alpha = 1 - confidence_level


def proportion_with_ci(successes, total, alpha=alpha, method=ci_method):
    if total == 0:
        return np.nan, np.nan, np.nan

    estimate = successes / total

    ci_low, ci_high = proportion_confint(
        count=successes,
        nobs=total,
        alpha=alpha,
        method=method,
    )

    return float(estimate), float(ci_low), float(ci_high)


def diagnostic_metrics_for_code(code):
    y_pred = patient_dx["pre_labels"].apply(lambda labels: code in labels).to_numpy()
    y_true = patient_dx["in_labels"].apply(lambda labels: code in labels).to_numpy()

    tp = int(np.sum(y_pred & y_true))
    fp = int(np.sum(y_pred & ~y_true))
    fn = int(np.sum(~y_pred & y_true))
    tn = int(np.sum(~y_pred & ~y_true))

    sensitivity = proportion_with_ci(tp, tp + fn)
    specificity = proportion_with_ci(tn, tn + fp)
    ppv = proportion_with_ci(tp, tp + fp)
    npv = proportion_with_ci(tn, tn + fn)
    accuracy = proportion_with_ci(tp + tn, tp + fp + fn + tn)

    return {
        "Diagnosis code": code,
        "Diagnosis": dx_code_to_name[code],
        "N": tp + fp + fn + tn,
        "Reference positive n": tp + fn,
        "Pre-hospital positive n": tp + fp,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Sensitivity": sensitivity[0],
        "Sensitivity CI low": sensitivity[1],
        "Sensitivity CI high": sensitivity[2],
        "Specificity": specificity[0],
        "Specificity CI low": specificity[1],
        "Specificity CI high": specificity[2],
        "PPV": ppv[0],
        "PPV CI low": ppv[1],
        "PPV CI high": ppv[2],
        "NPV": npv[0],
        "NPV CI low": npv[1],
        "NPV CI high": npv[2],
        "Accuracy": accuracy[0],
        "Accuracy CI low": accuracy[1],
        "Accuracy CI high": accuracy[2],
    }


diagnostic_metrics = pd.DataFrame(
    [diagnostic_metrics_for_code(code) for code in diagnosis_codes]
)

diagnostic_metrics

## Publication-style table

In [ ]:
def format_percent_ci(estimate, low, high, digits=1):
    if pd.isna(estimate):
        return "NA"

    scale = 100
    return (
        f"{estimate * scale:.{digits}f}% "
        f"({low * scale:.{digits}f}–{high * scale:.{digits}f}%)"
    )


publication_table = diagnostic_metrics[
    [
        "Diagnosis code",
        "Diagnosis",
        "Reference positive n",
        "Pre-hospital positive n",
        "TP",
        "FP",
        "FN",
        "TN",
    ]
].copy()

publication_table["Sensitivity (95% CI)"] = diagnostic_metrics.apply(
    lambda r: format_percent_ci(
        r["Sensitivity"], r["Sensitivity CI low"], r["Sensitivity CI high"]
    ),
    axis=1,
)

publication_table["Specificity (95% CI)"] = diagnostic_metrics.apply(
    lambda r: format_percent_ci(
        r["Specificity"], r["Specificity CI low"], r["Specificity CI high"]
    ),
    axis=1,
)

publication_table["PPV (95% CI)"] = diagnostic_metrics.apply(
    lambda r: format_percent_ci(r["PPV"], r["PPV CI low"], r["PPV CI high"]),
    axis=1,
)

publication_table["NPV (95% CI)"] = diagnostic_metrics.apply(
    lambda r: format_percent_ci(r["NPV"], r["NPV CI low"], r["NPV CI high"]),
    axis=1,
)

publication_table["Accuracy (95% CI)"] = diagnostic_metrics.apply(
    lambda r: format_percent_ci(
        r["Accuracy"], r["Accuracy CI low"], r["Accuracy CI high"]
    ),
    axis=1,
)

display(publication_table)

### Notes for interpretation

- In-hospital diagnoses are treated as the reference standard.
- PPV and NPV depend on the prevalence of each diagnosis in this study population.
- Rare diagnoses will naturally have wide confidence intervals.
- If there are no pre-hospital positive predictions for a diagnosis, PPV is undefined and is shown as `NA`.
- To report only trauma codes 1 through 11, change `diagnosis_codes` to `list(range(1, 12))`.
- To use exact Clopper-Pearson intervals instead of Wilson intervals, set `ci_method = "beta"` in the configuration cell.

## Optional: save the result tables

In [ ]:
# Uncomment if you want to save the numerical and publication-ready tables.
#
# publication_table.to_csv("/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/pediatric_trauma/comment/dx_diagnostic_metrics_publication_table.csv", index=False)